In [37]:
#data munapulation
import re
import numpy as np
import pandas as pd

#nltk
import nltk
nltk.download("stopwords",quiet=True)
nltk.download("punkt_tab",quiet=True)
nltk.download("punkt",quiet=True)

import emoji
from nltk.corpus import stopwords
from nltk.tokenize import TreebankWordTokenizer
from nltk.stem import PorterStemmer

#ml
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# visulization
import matplotlib.pyplot as plt
import seaborn as sns

In [38]:
data = pd.read_csv("data/amazon_reviews_dataset.csv")

In [39]:
data.shape

(600, 2)

In [40]:
print(data)

                                                               review_text  \
0    This laptop is absolutely amazing 😍 I have been using it for about...   
1    The headphones were delivered quickly but the sound quality is jus...   
2    I bought this blender for making my morning smoothies and it blend...   
3    These running shoes felt comfortable at first but they started fal...   
4    The coffee maker works fine I guess but it takes forever to brew a...   
..                                                                     ...   
595  The folding step stool wobbles alarmingly under normal adult weigh...   
596  My reading before bed has increased since using this adjustable bo...   
597  The seed starter pellets developed mould within a week in conditio...   
598  I have been oil pulling with this coconut blend for three months a...   
599  The recipe card box arrived with a crack along the spine hinge tha...   

     sentiment  
0            1  
1            0  
2           

In [41]:
data.dtypes

review_text      str
sentiment      int64
dtype: object

In [42]:
data['sentiment'].unique()

array([1, 0])

In [43]:
data.head(4)

,review_text,sentiment
0,This laptop is absolutely amazing 😍 I have been using it for about...,1
1,The headphones were delivered quickly but the sound quality is jus...,0
2,I bought this blender for making my morning smoothies and it blend...,1
3,These running shoes felt comfortable at first but they started fal...,0


In [44]:
data['sentiment'].value_counts()

sentiment
1    300
0    300
Name: count, dtype: int64

In [45]:
data['sentiment'] == 1

0       True
1      False
2       True
3      False
4      False
       ...  
595    False
596     True
597    False
598     True
599    False
Name: sentiment, Length: 600, dtype: bool

In [46]:
data[data['sentiment'] == 1].head(5)

,review_text,sentiment
0,This laptop is absolutely amazing 😍 I have been using it for about...,1
2,I bought this blender for making my morning smoothies and it blend...,1
5,Honestly this phone case is way better than I had expected 🎉 It fi...,1
7,This moisturizer has genuinely transformed my skin over the past m...,1
9,The wireless charger works as advertised but it charges my phone n...,1


In [47]:
sample_text = "The bookshelf requires two people to assemble but only includes instructions for one 😠 We spent nearly four hours assembling what should have been a straightforward task, spending $89.00 on this was. I wrote to info@techgadgetpro.net and got a quick reply. product link here https://www.amazon.com/dp/B0BX7HK."

In [48]:
def clean_tweet_1(text: str) -> str:
    """
    Step 1 — Structural noise removal.

    Handles every noisy token injected into the dataset:
      • Converts emojis → their English description (e.g. 😍 → 'smiling face')
        so sentiment signal is KEPT rather than silently dropped.
      • Strips URLs           (https://... or http://...)
      • Strips email addrs    (word@domain.ext)
      • Strips @handles       (@Username)
      • Strips prices         ($12.99 → removed)
      • Lowercases everything
      • Removes remaining punctuation and digits
      • Collapses multiple spaces to one
    """
    # 1a. Convert emojis to text descriptions BEFORE lowercasing
    #     emoji.demojize turns 😍 into ':smiling_face_with_heart-eyes:'
    # text = emoji.demojize(text, delimiters=(' ', ' '))

    # 1b. Lowercase everything
    text = text.lower()

    # 1c. Remove URLs  (must come before email to avoid partial matches)
    text = re.sub(r'https?://\S+', '', text)

    # 1d. Remove email addresses
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '', text)

    # 1e. Remove @handles
    text = re.sub(r'@\w+', '', text)

    # 1f. Remove prices like $12.99 or $199
    text = re.sub(r'\$\d+[\.,]?\d*', '', text)

    # 1g. Remove remaining punctuation, digits, and special characters
    #     Keep only alphabetic characters and spaces
    text = re.sub(r'[^a-z\s]', '', text)

    # 1h. Collapse multiple whitespace / newlines into a single space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [49]:
STOP_WORDS = set(stopwords.words('english'))
NEGATION_WORDS = {'no', 'not', "n't", 'nor', 'never', 'neither',
                  'without', 'hardly', 'barely', 'scarcely'}
STOP_WORDS -= NEGATION_WORDS   # do NOT remove negations


def clean_tweet_2(text: str) -> str:

    tokens = text.split()                              # simple whitespace split
    filtered = [w for w in tokens if w not in STOP_WORDS]
    return ' '.join(filtered)

In [50]:
TOKENIZER = TreebankWordTokenizer()

def tokenization(text: str) -> list:
    """
    Step 3 — Tokenization.

    Uses NLTK's TreebankWordTokenizer which:
      • Splits on whitespace and punctuation boundaries
      • Handles contractions correctly (don't → do, n't)
      • Preserves hyphenated compounds

    Returns a list of string tokens.
    """
    tokens = TOKENIZER.tokenize(text)
    # Drop any leftover single-character tokens (artefacts after cleaning)
    tokens = [t for t in tokens if len(t) > 1]
    return tokens

In [51]:
# Instantiate stemmer once
STEMMER = PorterStemmer()


def stem_tokens(tokens: list) -> list:
    """
    Step 4 — Stemming.

    Applies Porter Stemmer to each token.
    Examples:
      'running'  → 'run'
      'genuinely'→ 'genuinli'
      'amazing'  → 'amaz'

    Returns a list of stemmed string tokens.
    """
    return [STEMMER.stem(token) for token in tokens]

In [52]:
def preprocess(text: str) -> str:
    """
    Master pipeline — applies every preprocessing step in order
    and returns a single cleaned string ready for TF-IDF vectorisation.

    Pipeline
    ────────
    raw text
      │
      ▼  clean_tweet_1()  ── emoji→text, lowercase, strip URLs/emails/
      │                       handles/prices, remove punct & digits
      ▼  clean_tweet_2()  ── remove English stopwords (keep negations)
      ▼  tokenization()   ── Treebank word tokenizer → list of tokens
      ▼  stem_tokens()    ── Porter Stemmer on each token
      ▼  join(' ')        ── reassemble into a single string
      │
    cleaned string
    """
    text = clean_tweet_1(text)      # Step 1: noise removal
    text = clean_tweet_2(text)      # Step 2: stopword removal
    tokens = tokenization(text)     # Step 3: tokenization
    tokens = stem_tokens(tokens)    # Step 4: stemming
    return ' '.join(tokens)         # reassemble for TF-IDF

In [53]:
def preprocess(text: str) -> str:
    """
    Master pipeline — applies every preprocessing step in order
    and returns a single cleaned string ready for TF-IDF vectorisation.

    Pipeline
    ────────
    raw text
      │
      ▼  clean_tweet_1()  ── emoji→text, lowercase, strip URLs/emails/
      │                       handles/prices, remove punct & digits
      ▼  clean_tweet_2()  ── remove English stopwords (keep negations)
      ▼  tokenization()   ── Treebank word tokenizer → list of tokens
      ▼  stem_tokens()    ── Porter Stemmer on each token
      ▼  join(' ')        ── reassemble into a single string
      │
    cleaned string
    """
    text = clean_tweet_1(text)      # Step 1: noise removal
    text = clean_tweet_2(text)      # Step 2: stopword removal
    tokens = tokenization(text)     # Step 3: tokenization
    tokens = stem_tokens(tokens)    # Step 4: stemming
    return ' '.join(tokens)         # reassemble for TF-IDF

In [54]:
print('═' * 65)
print('STEP-BY-STEP PREPROCESSING DEMO')
print('═' * 65)

print(f"\n[RAW]       {sample_text}")

after_step1 = clean_tweet_1(sample_text)
print(f"\n[STEP 1]    {after_step1}")
print("             ↳ emojis demojized, URLs/emails/handles/$prices removed, lowercased")

after_step2 = clean_tweet_2(after_step1)
print(f"\n[STEP 2]    {after_step2}")
print("             ↳ stopwords removed (negations kept)")

after_step3 = tokenization(after_step2)
print(f"\n[STEP 3]    {after_step3}")
print("             ↳ Treebank tokenized → list of tokens")

after_step4 = stem_tokens(after_step3)
print(f"\n[STEP 4]    {after_step4}")
print("             ↳ Porter Stemmer applied to each token")

final = ' '.join(after_step4)
print(f"\n[FINAL]     {final}")
print('═' * 65)

═════════════════════════════════════════════════════════════════
STEP-BY-STEP PREPROCESSING DEMO
═════════════════════════════════════════════════════════════════

[RAW]       The bookshelf requires two people to assemble but only includes instructions for one 😠 We spent nearly four hours assembling what should have been a straightforward task, spending $89.00 on this was. I wrote to info@techgadgetpro.net and got a quick reply. product link here https://www.amazon.com/dp/B0BX7HK.

[STEP 1]    the bookshelf requires two people to assemble but only includes instructions for one we spent nearly four hours assembling what should have been a straightforward task spending on this was i wrote to and got a quick reply product link here
             ↳ emojis demojized, URLs/emails/handles/$prices removed, lowercased

[STEP 2]    bookshelf requires two people assemble includes instructions one spent nearly four hours assembling straightforward task spending wrote got quick reply product link
 

In [55]:
print('Applying preprocessing pipeline to all 300 reviews...')
data['cleaned_text'] = data['review_text'].apply(preprocess)

print(f'Done ✅  — cleaned_text column added')
print(f'\nAvg tokens before cleaning: {data["review_text"].str.split().str.len().mean():.1f}')
print(f'Avg tokens after  cleaning: {data["cleaned_text"].str.split().str.len().mean():.1f}')

# Side-by-side comparison
comparison = data[['review_text', 'cleaned_text']].head(5)
pd.set_option('display.max_colwidth', 70)
comparison

Applying preprocessing pipeline to all 300 reviews...
Done ✅  — cleaned_text column added

Avg tokens before cleaning: 35.0
Avg tokens after  cleaning: 19.0


,review_text,cleaned_text
0,This laptop is absolutely amazing 😍 I have been using it for about...,laptop absolut amaz use three week perform total outstand daili ta...
1,The headphones were delivered quickly but the sound quality is jus...,headphon deliv quickli sound qualiti okay honestli expect much bet...
2,I bought this blender for making my morning smoothies and it blend...,bought blender make morn smoothi blend everyth perfectli smooth mo...
3,These running shoes felt comfortable at first but they started fal...,run shoe felt comfort first start fall apart week regular use stit...
4,The coffee maker works fine I guess but it takes forever to brew a...,coffe maker work fine guess take forev brew full pot old machin us...


In [56]:
data.columns

Index(['review_text', 'sentiment', 'cleaned_text'], dtype='str')

In [57]:
X=data['cleaned_text']
y=data['sentiment']

In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  #for small dataset, we can use 20% for testing and 80% for training

In [59]:
tfidf = TfidfVectorizer() #text to vector, text vectorization, text to numeric vector, text to number vector, text to feature vector

In [60]:
X_tfidf = tfidf.fit_transform(X)
X_train_tfidf = tfidf.transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [61]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [62]:
model.score(X_test_tfidf, y_test)

0.9916666666666667

In [63]:
y_pred = model.predict(X_test_tfidf) # text data

In [64]:
accuracy = accuracy_score(y_test, y_pred)

In [65]:
accuracy_score

<function sklearn.metrics._classification.accuracy_score(y_true, y_pred, *, normalize=True, sample_weight=None)>

In [66]:
classification_report(y_test, y_pred, target_names=['Negative', 'Positive'])

'              precision    recall  f1-score   support\n\n    Negative       0.98      1.00      0.99        58\n    Positive       1.00      0.98      0.99        62\n\n    accuracy                           0.99       120\n   macro avg       0.99      0.99      0.99       120\nweighted avg       0.99      0.99      0.99       120\n'

In [67]:
#.....Deploying the model......

In [68]:
brand_new_review = """This portable blender is hands down the best purchase I made this year 🥤 at $39.99 it outperforms machines triple the price. check the listing at https://amzn.to/3xYqP1r. @HomeChefPro called it a game changer and I completely agree."""

In [69]:
clean_review = preprocess(brand_new_review)

In [70]:
clean_review_tfidf = tfidf.transform([clean_review])

In [71]:
single_review_prediction = model.predict(clean_review_tfidf)

In [72]:
if single_review_prediction[0] == 1:
    print("Positive review")
else:
    print("Negative review") 

Negative review
